In [291]:
import requests
from datetime import datetime
import json

# An api key is emailed to you when you sign up to a plan
# Get a free API key at https://api.the-odds-api.com/
API_KEY = 'aebd96cf4102e7110abbfad4f9a5e3a0'

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#
# First get a list of in-season sports
#   The sport 'key' from the response can be used to get odds in the next request
#
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
SPORT = 'basketball_nba'
sports_response = requests.get(
    f'https://api.the-odds-api.com/v4/sports/{SPORT}/events', 
    params={
        'api_key': API_KEY
    }
)


if sports_response.status_code != 200:
    print(f'Failed to get sports: status_code {sports_response.status_code}, response body {sports_response.text}')

else:
    print('List of in season sports:', sports_response.json())
    sports_data = sports_response.json()
    pretty_data = json.dumps(sports_data, indent=4)
    print(pretty_data)

# Extracting the event IDs into a list
event_ids_list = [event['id'] for event in sports_data]

List of in season sports: [{'id': '3bc5054ad4bc0f2c50cd87a3fcb8cd45', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2025-04-24T23:05:00Z', 'home_team': 'Detroit Pistons', 'away_team': 'New York Knicks'}, {'id': 'f39fff52133794a10c16ba61d2c0eeda', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2025-04-25T01:35:00Z', 'home_team': 'Memphis Grizzlies', 'away_team': 'Oklahoma City Thunder'}, {'id': '19aca5a6e1fa8b1aa68505d3f1d75b37', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2025-04-25T02:05:00Z', 'home_team': 'Los Angeles Clippers', 'away_team': 'Denver Nuggets'}, {'id': '5a935d4f284b3fcf84515f522e9079f9', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2025-04-25T23:05:00Z', 'home_team': 'Orlando Magic', 'away_team': 'Boston Celtics'}, {'id': '8af134619ff871a0b76992d76de7e224', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2025-04-26T00:05:00Z', 'home_team': 'Milwa

In [292]:



def get_game_odds(event_ids, API_KEY=API_KEY, sport='basketball_nba', regions='us', markets='player_points', odds_format='decimal', date_format='iso', time_from='2025-02-10T00:00:00Z', time_to='2025-05-10T00:00:00Z'):
    # Convert input times to ISO format
    iso_time_from = datetime.strptime(time_from, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%dT%H:%M:%SZ")
    iso_time_to = datetime.strptime(time_to, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%dT%H:%M:%SZ")
    
    # Initialize a list to store data for all events
    all_games_data = []

    for event_id in event_ids:
        # Make the request to get odds data for the event
        odds_response = requests.get(
            f'https://api.the-odds-api.com/v4/sports/{sport}/events/{event_id}/odds',
            params={
                'api_key': API_KEY,
                'regions': regions,
                'markets': markets,
                'oddsFormat': odds_format,
                'dateFormat': date_format,
                'commenceTimeFrom': iso_time_from,
                'commenceTimeTo': iso_time_to
            }
        )

        if odds_response.status_code != 200:
            print(f'Failed to get odds for EventID {event_id}: status_code {odds_response.status_code}, response body {odds_response.text}')
        else:
            odds_json = odds_response.json()
            print(f'Number of events for EventID {event_id}:', len(odds_json))
            all_games_data.append(odds_json)
            
            # Save the response to a file for each event
            with open(f"games_data_{event_id}.json", "w") as file:
                json.dump(odds_json, file)

            # Check the usage quota
            print(f'Remaining requests for EventID {event_id}:', odds_response.headers['x-requests-remaining'])
            print(f'Used requests for EventID {event_id}:', odds_response.headers['x-requests-used'])

    return all_games_data

event_ids = event_ids_list

games_data = get_game_odds(event_ids)
with open("games_data.json", "w") as file:
    json.dump(games_data, file)


Number of events for EventID 3bc5054ad4bc0f2c50cd87a3fcb8cd45: 7
Remaining requests for EventID 3bc5054ad4bc0f2c50cd87a3fcb8cd45: 217
Used requests for EventID 3bc5054ad4bc0f2c50cd87a3fcb8cd45: 283
Number of events for EventID f39fff52133794a10c16ba61d2c0eeda: 7
Remaining requests for EventID f39fff52133794a10c16ba61d2c0eeda: 216
Used requests for EventID f39fff52133794a10c16ba61d2c0eeda: 284
Number of events for EventID 19aca5a6e1fa8b1aa68505d3f1d75b37: 7
Remaining requests for EventID 19aca5a6e1fa8b1aa68505d3f1d75b37: 215
Used requests for EventID 19aca5a6e1fa8b1aa68505d3f1d75b37: 285
Number of events for EventID 5a935d4f284b3fcf84515f522e9079f9: 7
Remaining requests for EventID 5a935d4f284b3fcf84515f522e9079f9: 214
Used requests for EventID 5a935d4f284b3fcf84515f522e9079f9: 286
Number of events for EventID 8af134619ff871a0b76992d76de7e224: 7
Remaining requests for EventID 8af134619ff871a0b76992d76de7e224: 213
Used requests for EventID 8af134619ff871a0b76992d76de7e224: 287
Number of 

In [293]:
import pandas as pd
pd.set_option('display.max_columns', None)

with open('games_data.json', 'r') as file:
    data = json.load(file)

#print(data)
pretty_data = json.dumps(data, indent=4)
print(pretty_data)

[
    {
        "id": "3bc5054ad4bc0f2c50cd87a3fcb8cd45",
        "sport_key": "basketball_nba",
        "sport_title": "NBA",
        "commence_time": "2025-04-24T23:05:00Z",
        "home_team": "Detroit Pistons",
        "away_team": "New York Knicks",
        "bookmakers": [
            {
                "key": "draftkings",
                "title": "DraftKings",
                "markets": [
                    {
                        "key": "player_points",
                        "last_update": "2025-04-24T16:17:28Z",
                        "outcomes": [
                            {
                                "name": "Over",
                                "description": "Jalen Brunson",
                                "price": 1.87,
                                "point": 28.5
                            },
                            {
                                "name": "Under",
                                "description": "Jalen Brunson",
                     

In [294]:
from datetime import datetime
import pytz
def extract_data(data):
    extracted_data = []
    utc_zone = pytz.timezone('UTC')
    eastern_zone = pytz.timezone('US/Eastern')

    # Loop through each entry in the JSON
    for data in data:
        match_id = data['id']
        home_team = data['home_team']
        away_team = data['away_team']
        for bookmaker in data['bookmakers']:
            bookmaker_title = bookmaker['title']
            print(bookmaker_title)
            for market in bookmaker['markets']:
                market_title = market['key']
                print(market_title)
                for outcome in market['outcomes']:
                    OverOrUnder = outcome['name']
                    player_name = outcome['description']
                    outcome_price = outcome['price']
                    points = outcome['point']
                    #outcome_datetime = datetime.fromtimestamp(outcome['timestamp'], utc_zone).astimezone(eastern_zone)
                    #outcome_datetime = outcome_datetime.strftime('%Y-%m-%d %H:%M:%S')
                    extracted_data.append({
                        'ID': match_id,
                        #'StartDate': outcome_datetime,  
                        'HomeTeam': home_team,
                        'AwayTeam': away_team,
                        'Bookmaker': bookmaker_title,
                        'Market': market_title,
                        'Player': player_name,
                        'Outcome': OverOrUnder,
                        'Points': points,
                        'Price': outcome_price,
                    })
    df = pd.DataFrame(extracted_data)
    df
    return df

df = extract_data(data)
df

DraftKings
player_points
FanDuel
player_points
BetMGM
player_points
BetOnline.ag
player_points
Bovada
player_points
BetRivers
player_points
DraftKings
player_points
FanDuel
player_points
BetOnline.ag
player_points
BetMGM
player_points
BetRivers
player_points
Bovada
player_points
DraftKings
player_points
FanDuel
player_points
BetMGM
player_points
BetOnline.ag
player_points
BetRivers
player_points
Bovada
player_points
DraftKings
player_points
FanDuel
player_points
BetMGM
player_points
BetOnline.ag
player_points
Bovada
player_points
DraftKings
player_points
BetOnline.ag
player_points
BetMGM
player_points
FanDuel
player_points
Bovada
player_points
BetRivers
player_points
FanDuel
player_points
DraftKings
player_points
BetRivers
player_points
BetOnline.ag
player_points
BetMGM
player_points
Bovada
player_points
DraftKings
player_points
FanDuel
player_points
BetOnline.ag
player_points
BetMGM
player_points
Bovada
player_points
BetMGM
player_points
Bovada
player_points


,ID,HomeTeam,AwayTeam,Bookmaker,Market,Player,Outcome,Points,Price
0,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Over,28.5,1.87
1,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Under,28.5,1.87
2,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Over,28.5,1.91
3,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Under,28.5,1.83
4,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Karl-Anthony Towns,Over,22.5,1.95
...,...,...,...,...,...,...,...,...,...
1113,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Fred VanVleet,Under,12.5,1.91
1114,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Over,20.5,1.95
1115,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Under,20.5,1.80
1116,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Tari Eason,Over,8.5,1.95


In [295]:
df['adjusted_price'] = ((df['Price'] * 1) - 1)

In [296]:
def decimal_to_us_odds(decimal_odds):
    if decimal_odds == 1.00:
        return 0  # No payout, return 0 or another meaningful value
    elif decimal_odds >= 2:
        # Underdogs: (decimal_odds - 1) * 100
        return (decimal_odds - 1) * 100
    else:
        # Favorites: -100 / (decimal_odds - 1)
        return -100 / (decimal_odds - 1)

In [297]:
df['US_Odds'] = df['Price'].apply(decimal_to_us_odds)
df

,ID,HomeTeam,AwayTeam,Bookmaker,Market,Player,Outcome,Points,Price,adjusted_price,US_Odds
0,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Over,28.5,1.87,0.87,-114.942529
1,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Under,28.5,1.87,0.87,-114.942529
2,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Over,28.5,1.91,0.91,-109.890110
3,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Under,28.5,1.83,0.83,-120.481928
4,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Karl-Anthony Towns,Over,22.5,1.95,0.95,-105.263158
...,...,...,...,...,...,...,...,...,...,...,...
1113,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Fred VanVleet,Under,12.5,1.91,0.91,-109.890110
1114,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Over,20.5,1.95,0.95,-105.263158
1115,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Under,20.5,1.80,0.80,-125.000000
1116,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Tari Eason,Over,8.5,1.95,0.95,-105.263158


In [298]:
df.to_excel('player_points_data.xlsx', index=False)

In [299]:
#Limit the sportsbooks to only my avaliable sportsbooks. 
accepted_sportsbooks = ['FanDuel', 'DraftKings', 'BetMGM', 'Caesars','BetRivers']

accceptedsportsbooksdf = df[df['Bookmaker'].isin(accepted_sportsbooks)]
df

,ID,HomeTeam,AwayTeam,Bookmaker,Market,Player,Outcome,Points,Price,adjusted_price,US_Odds
0,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Over,28.5,1.87,0.87,-114.942529
1,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Under,28.5,1.87,0.87,-114.942529
2,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Over,28.5,1.91,0.91,-109.890110
3,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Under,28.5,1.83,0.83,-120.481928
4,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Karl-Anthony Towns,Over,22.5,1.95,0.95,-105.263158
...,...,...,...,...,...,...,...,...,...,...,...
1113,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Fred VanVleet,Under,12.5,1.91,0.91,-109.890110
1114,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Over,20.5,1.95,0.95,-105.263158
1115,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Jalen Green,Under,20.5,1.80,0.80,-125.000000
1116,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,Bovada,player_points,Tari Eason,Over,8.5,1.95,0.95,-105.263158


In [300]:
# First, drop duplicates on the key columns
unique_combinations = accceptedsportsbooksdf[['Player', 'Outcome', 'Points']].drop_duplicates()

# Split into Over and Under
overs = unique_combinations[unique_combinations['Outcome'] == 'Over']
unders = unique_combinations[unique_combinations['Outcome'] == 'Under']

# Convert to list of tuples (or dicts, if you prefer)
over_list = list(overs.itertuples(index=False, name=None))
under_list = list(unders.itertuples(index=False, name=None))

print(over_list)
overs

[('Jalen Brunson', 'Over', 28.5), ('Cade Cunningham', 'Over', 28.5), ('Karl-Anthony Towns', 'Over', 22.5), ('OG Anunoby', 'Over', 16.5), ('Mikal Bridges', 'Over', 15.5), ('Malik Beasley', 'Over', 14.5), ('Tobias Harris', 'Over', 14.5), ('Jalen Duren', 'Over', 11.5), ('Josh Hart', 'Over', 11.5), ('Dennis Schroder', 'Over', 10.5), ('Ausar Thompson', 'Over', 9.5), ('Tim Hardaway Jr', 'Over', 9.5), ('Mitchell Robinson', 'Over', 3.5), ('Karl-Anthony Towns', 'Over', 21.5), ('Jalen Brunson', 'Over', 29.5), ('Isaiah Stewart II', 'Over', 4.5), ('Miles McBride', 'Over', 4.5), ('Cade Cunningham', 'Over', 26.5), ('Ausar Thompson', 'Over', 10.5), ('Malik Beasley', 'Over', 15.5), ('Mikal Bridges', 'Over', 16.5), ('Dennis Schroder', 'Over', 9.5), ('Jalen Duren', 'Over', 12.5), ('Cade Cunningham', 'Over', 27.5), ('Josh Hart', 'Over', 12.5), ('OG Anunoby', 'Over', 15.5), ('OG Anunoby', 'Over', 17.5), ('Mikal Bridges', 'Over', 17.5), ('Tobias Harris', 'Over', 13.5), ('Malik Beasley', 'Over', 16.5), ('Ja

,Player,Outcome,Points
0,Jalen Brunson,Over,28.5
2,Cade Cunningham,Over,28.5
4,Karl-Anthony Towns,Over,22.5
6,OG Anunoby,Over,16.5
8,Mikal Bridges,Over,15.5
...,...,...,...
1096,Tari Eason,Over,8.5
1098,Steven Adams,Over,3.5
1100,Dillon Brooks,Over,11.5
1102,Jalen Green,Over,20.5


In [301]:
#For a specified team in the list find the highest US Odds
def find_highest_us_odds_for_player_Over(df, Player, Points):
    # Filter the dataframe for the specified team
    player_df = df[(df['Player'] == Player) & (df['Outcome'] == 'Over') & (df['Points'] == Points)]
    if not player_df.empty:
        return player_df.loc[[player_df['US_Odds'].idxmax()]]
    else:
        return None

find_highest_us_odds_for_player_Over(accceptedsportsbooksdf, 'Darius Garland', 19.5)

#For a specified team in the list find the highest US Odds
def find_highest_us_odds_for_player_Under(df, Player, Points):
    # Filter the dataframe for the specified team
    player_df = df[(df['Player'] == Player) & (df['Outcome'] == 'Under') & (df['Points'] == Points)]
    if not player_df.empty:
        return player_df.loc[[player_df['US_Odds'].idxmax()]]
    else:
        return None

find_highest_us_odds_for_player_Under(accceptedsportsbooksdf, 'Darius Garland', 19.5)

In [302]:
results = []  # Use a list instead of a DataFrame

# Loop through all Over combinations
for line in over_list:
    player, outcome, points = line
    maxvaluedf = find_highest_us_odds_for_player_Over(accceptedsportsbooksdf, player, points)
    results.append(maxvaluedf)

# Combine into a single DataFrame
final_df_over = pd.concat(results, ignore_index=True)

# View the result
print(type(final_df_over))  # <class 'pandas.core.frame.DataFrame'>
final_df_over



<class 'pandas.core.frame.DataFrame'>


,ID,HomeTeam,AwayTeam,Bookmaker,Market,Player,Outcome,Points,Price,adjusted_price,US_Odds
0,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,FanDuel,player_points,Jalen Brunson,Over,28.5,1.93,0.93,-107.526882
1,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,BetRivers,player_points,Cade Cunningham,Over,28.5,1.93,0.93,-107.526882
2,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,BetRivers,player_points,Karl-Anthony Towns,Over,22.5,1.97,0.97,-103.092784
3,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,FanDuel,player_points,OG Anunoby,Over,16.5,1.94,0.94,-106.382979
4,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Mikal Bridges,Over,15.5,1.80,0.80,-125.000000
...,...,...,...,...,...,...,...,...,...,...,...
204,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Tari Eason,Over,8.5,1.95,0.95,-105.263158
205,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Steven Adams,Over,3.5,1.74,0.74,-135.135135
206,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Dillon Brooks,Over,11.5,1.87,0.87,-114.942529
207,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Jalen Green,Over,20.5,1.95,0.95,-105.263158


In [303]:
results = []  # Use a list instead of a DataFrame

# Loop through all Over combinations
for line in under_list:
    player, outcome, points = line
    maxvaluedf = find_highest_us_odds_for_player_Under(accceptedsportsbooksdf, player, points)
    results.append(maxvaluedf)

# Combine into a single DataFrame
final_df_under = pd.concat(results, ignore_index=True)

# View the result
print(type(final_df_under))  # <class 'pandas.core.frame.DataFrame'>
final_df_under

<class 'pandas.core.frame.DataFrame'>


,ID,HomeTeam,AwayTeam,Bookmaker,Market,Player,Outcome,Points,Price,adjusted_price,US_Odds
0,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Jalen Brunson,Under,28.5,1.87,0.87,-114.942529
1,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,DraftKings,player_points,Cade Cunningham,Under,28.5,1.83,0.83,-120.481928
2,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,FanDuel,player_points,Karl-Anthony Towns,Under,22.5,1.85,0.85,-117.647059
3,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,BetRivers,player_points,OG Anunoby,Under,16.5,1.88,0.88,-113.636364
4,3bc5054ad4bc0f2c50cd87a3fcb8cd45,Detroit Pistons,New York Knicks,BetRivers,player_points,Mikal Bridges,Under,15.5,2.08,1.08,108.000000
...,...,...,...,...,...,...,...,...,...,...,...
204,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Tari Eason,Under,8.5,1.80,0.80,-125.000000
205,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Steven Adams,Under,3.5,2.00,1.00,100.000000
206,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Dillon Brooks,Under,11.5,1.87,0.87,-114.942529
207,64727ecb83898fd5fa4ba086945dd594,Golden State Warriors,Houston Rockets,BetMGM,player_points,Jalen Green,Under,20.5,1.80,0.80,-125.000000


In [304]:
final_df_under.to_excel('player_points_results.xlsx', index=False)

In [305]:
merged_df = pd.merge(final_df_over, final_df_under, on=['Player', 'Points'], how='inner')

In [306]:
merged_df.to_excel('player_points_results.xlsx', index=False)